*Downloaded from UC Berkeley Data 100, course notes Fall 2023, "SQL I" ([source](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd), BSD 3-Clause licence, Copyright (c) 2022 UC Berkeley Data 100). Course changes: converted from Quarto to a notebook, and the `%%sql` cells rewritten as `pd.read_sql_query(...)` so that it runs in the course environment; see [README.md](README.md).*

> **Important Note.**
> **This course note was developed in Fall 2023. If you are taking this class in a future semester, please keep in mind that this note may not be up to date with course content for that semester.**

> **Learning Outcomes.**
> * Identify situations where a database may be preferred over a CSV file
> * Write basic SQL queries using `SELECT`, `FROM`, `WHERE`, `ORDER BY`, `LIMIT`, and `OFFSET`
> * Perform aggregations using `GROUP BY`

So far in the course, we have made our way through the entire data science lifecycle: we learned how to load and explore a dataset, formulate questions, and use the tools of prediction and inference to come up with answers. For the remaining weeks of the semester, we are going to make a second pass through the lifecycle, this time doing so with a different set of tools, ideas, and abstractions. 

## Databases
With this goal in mind, let's go back to the very beginning of the lifecycle. We first started our work in data analysis by looking at the `pandas` library, which offered us powerful tools to manipulate tabular data stored in (primarily) CSV files. CSVs work well when analyzing relatively small datasets that don't need to be shared across many users. In research and industry, however, data scientists often need to access enormous bodies of data that cannot be easily stored in a CSV format. Collaborating with others when working with CSVs can also be tricky – a real-world data scientist may run into problems when multiple users try to make modifications, or, even worse, security issues with who should and should not have access to the data. 

A **database** is a large, organized collection of data. Databases are administered by **Database Management Systems (DBMS)**, which are software systems that store, manage, and facilitate access to one or more databases. Databases help mitigate many of the issues that come with using CSVs for data storage: they provide reliable storage that can survive system crashes or disk failures, are optimized to compute on data that does not fit into memory, and contain special data structures to improve performance. Using databases rather than CSVs offers further benefits from the standpoint of data management. A DBMS can apply settings that configure how data is organized, block certain data anomalies (for example, enforcing non-negative weights or ages), and determine who is allowed access to the data. It can also ensure safe concurrent operations where multiple users reading and writing to the database will not lead to fatal errors.

As you may have guessed, we can't use our usual `pandas` methods to work with data in a database. Instead, we'll turn to Structured Query Language.

## Structured Query Language and Database Schema

**Structured Query Language**, or **SQL** (commonly pronounced "sequel," though this is the subject of [fierce debate](https://patorjk.com/blog/2012/01/26/pronouncing-sql-s-q-l-or-sequel/)), is a special programming language designed to communicate with databases. You may have encountered it in classes like CS 61A or Data C88C before. Unlike Python, it is a **declarative programming language** – this means that rather than writing the exact logic needed to complete a task, a piece of SQL code "declares" what the desired final output should be and leaves the program to determine what logic should be implemented. 

It is important to reiterate that SQL is an entirely different language from Python. However, Python *does* have special engines that allow us to run SQL code in a Jupyter notebook. While this is typically not how SQL is used outside of an educational setting, we will be using this workflow to illustrate how SQL queries are constructed using the tools we've already worked with this semester. You will learn more about how to run SQL queries in Jupyter in Lab 10.

The syntax below will seem unfamiliar to you; for now, just focus on understanding the output displayed. We will clarify the SQL code in a bit.

To start, we'll look at a database called `basic_examples.db`.

In [1]:
# Load the SQL Alchemy Python library
import sqlalchemy
import pandas as pd

In [2]:
# Course version: plain sqlite3 instead of the %sql magic
import sqlite3

Connect to the SQLite database `basic_examples.db`.

In [3]:
# Course version: connect with sqlite3
con = sqlite3.connect("data/basic_examples.db")

In [4]:
pd.read_sql_query("""
SELECT * 
FROM sqlite_master
WHERE type="table"
""", con)

,type,name,tbl_name,rootpage,sql
0,table,sqlite_sequence,sqlite_sequence,7,"CREATE TABLE sqlite_sequence(name,seq)"
1,table,Dragon,Dragon,2,CREATE TABLE Dragon (\n name TEXT PRIMARY K...
2,table,Dish,Dish,4,CREATE TABLE Dish (\n name TEXT PRIMARY KEY...
3,table,Scene,Scene,6,CREATE TABLE Scene (\n id INTEGER PRIMARY K...


The summary above displays information about the database. The database contains four tables, named `sqlite_sequence`, `Dragon`, `Dish`, and `Scene`. The rightmost column above lists the command that was used to construct each table. 

Let's look more closely at the command used to create the `Dragon` table (the second entry above). 

    CREATE TABLE Dragon (name TEXT PRIMARY KEY,
                         year INTEGER CHECK (year >= 2000),
                         cute INTEGER)

The statement `CREATE TABLE` is used to specify the **schema** of the table – a description of what logic is used to organize the table. Schema follows a set format:

* `ColName`: the name of a column
* `DataType`: the type of data to be stored in a column. Some of the most common SQL data types are `INT` (integers), `FLOAT` (floating point numbers), `TEXT` (strings), `BLOB` (arbitrary data, such as audio/video files), and `DATETIME` (a date and time).
* `Constraint`: some restriction on the data to be stored in the column. Common constraints are `CHECK` (data must obey a certain condition), `PRIMARY KEY` (designate a column as the table's primary key), `NOT NULL` (data cannot be null), and `DEFAULT` (a default fill value if no specific entry is given).

We see that `Dragon` contains three columns. The first of these, `"name"`, contains text data. It is designated as the **primary key** of the table; that is, the data contained in `"name"` uniquely identifies each entry in the table. Because `"name"` is the primary key of the table, no two entries in the table can have the same name – a given value of `"name"` is unique to each dragon. The `"year"` column contains integer data, with the constraint that year values must be greater than or equal to 2000. The final column, `"cute"`, contains integer data with no restrictions on allowable values. 

We can verify this by viewing `Dragon` itself.

In [5]:
pd.read_sql_query("""
SELECT *
FROM Dragon
""", con)

,name,year,cute
0,hiccup,2010,10
1,drogon,2011,-100
2,dragon 2,2019,0


Database tables (also referred to as **relations**) are structured much like `DataFrame`s in `pandas`. Each row, sometimes called a **tuple**, represents a single record in the dataset. Each column, sometimes called an **attribute** or **field**, describes some feature of the record. 

## `SELECT`ing From Tables

To extract and manipulate data stored in a SQL table, we will need to familiarize ourselves with the syntax to write pieces of SQL code, which we call **queries**. 

The basic unit of a SQL query is the `SELECT` statement. `SELECT` specifies what columns we would like to extract from a given table. We use `FROM` to tell SQL the table from which we want to `SELECT` our data.

In [6]:
pd.read_sql_query("""
SELECT *
FROM Dragon
""", con)

,name,year,cute
0,hiccup,2010,10
1,drogon,2011,-100
2,dragon 2,2019,0


In SQL, `*` means "everything." The query above grabs *all* the columns in `Dragon` and displays them in the outputted table. We can also specify a specific subset of columns to be `SELECT`ed. Notice that the outputted columns appear in the order that they were `SELECT`ed.

In [7]:
pd.read_sql_query("""
SELECT cute, year
FROM Dragon
""", con)

,cute,year
0,10,2010
1,-100,2011
2,0,2019


And just like that, we've already written two SQL queries. There are a few points of note in the queries above. Firstly, notice that every "verb" is written in uppercase. It is convention to write SQL operations in capital letters, but your code will run just fine even if you choose to keep things in lowercase. Second, the query above separates each statement with a new line. SQL queries are not impacted by whitespace within the query; this means that SQL code is typically written with a new line after each statement to make things more readable. The semicolon (`;`) indicates the end of a query. There are some "flavors" of SQL in which a query will not run if no semicolon is present; however, in Data 100, the SQL version we will use works with or without an ending semicolon. Queries in these notes will end with semicolons to build up good habits.

The `AS` keyword allows us to give a column a new name (called an **alias**) after it has been `SELECT`ed. The general syntax is:

    SELECT column_name_in_database_table AS new_name_in_output_table

In [8]:
pd.read_sql_query("""
SELECT cute AS cuteness, year AS birth
FROM Dragon
""", con)

,cuteness,birth
0,10,2010
1,-100,2011
2,0,2019


To `SELECT` only the *unique* values in a column, we use the `DISTINCT` keyword. This will cause any any duplicate entries in a column to be removed. If we want to find only the unique years in `Dragon`, without any repeats, we would write:

In [9]:
pd.read_sql_query("""
SELECT DISTINCT year
FROM Dragon
""", con)

,year
0,2010
1,2011
2,2019


**Every** SQL query must include both a `SELECT` and `FROM` statement. Intuitively, this makes sense – we know that we'll want to extract some piece of information from the table; to do so, we also need to indicate what table we want to consider. 

It is important to note that SQL enforces a strict "order of operations" – SQL clauses must *always* follow the same sequence. For example, the `SELECT` statement must always precede `FROM`. This means that any SQL query will follow the same structure. 

    SELECT <column list>
    FROM <table>
    [additional clauses]
    
The additional clauses that we use depend on the specific task trying to be achieved. We may refine our query to filter on a certain condition, aggregate a particular column, or join several tables together. We will spend the rest of this lecture outlining some useful clauses to build up our understanding of the order of operations.

## Applying `WHERE` Conditions

The `WHERE` keyword is used to select only some rows of a table, filtered on a given Boolean condition.

In [10]:
pd.read_sql_query("""
SELECT name, year
FROM Dragon
WHERE cute > 0
""", con)

,name,year
0,hiccup,2010


We can add complexity to the `WHERE` condition using the keywords `AND`, `OR`, and `NOT`, much like we would in Python.

In [11]:
pd.read_sql_query("""
SELECT name, year
FROM Dragon
WHERE cute > 0 OR year > 2013
""", con)

,name,year
0,hiccup,2010
1,dragon 2,2019


To spare ourselves needing to write complicated logical expressions by combining several conditions, we can also filter for entries that are `IN` a specified list of values. This is similar to the use of `in` or `.isin` in Python.

In [12]:
pd.read_sql_query("""
SELECT name, year
FROM Dragon
WHERE name IN ("hiccup", "puff")
""", con)

,name,year
0,hiccup,2010


You may have noticed earlier that our table actually has a missing value. In SQL, missing data is given the special value `NULL`. `NULL` behaves in a fundamentally different way to other data types. We can't use the typical operators (=, >, and <) on `NULL` values (in fact, `NULL == NULL` returns `False`!); instead, we check to see if a value `IS` or `IS NOT` `NULL`.

In [13]:
pd.read_sql_query("""
SELECT *
FROM Dragon
WHERE cute IS NOT NULL
""", con)

,name,year,cute
0,hiccup,2010,10
1,drogon,2011,-100
2,dragon 2,2019,0


## Sorting and Restricting Output
    
What if we want the output table to appear in a certain order? The `ORDER BY` keyword behaves similarly to `.sort_values()` in `pandas`.

In [14]:
pd.read_sql_query("""
SELECT *
FROM Dragon
ORDER BY cute
""", con)

,name,year,cute
0,drogon,2011,-100
1,dragon 2,2019,0
2,hiccup,2010,10


By default, `ORDER BY` will display results in ascending order (with the lowest values at the top of the table). To sort in descending order, we use the `DESC` keyword after specifying the column to be used for ordering.

In [15]:
pd.read_sql_query("""
SELECT *
FROM Dragon
ORDER BY cute DESC
""", con)

,name,year,cute
0,hiccup,2010,10
1,dragon 2,2019,0
2,drogon,2011,-100


We can also tell SQL to `ORDER BY` two columns at once. This will sort the table by the first listed column, then use the values in the second listed column to break any ties.

In [16]:
pd.read_sql_query("""
SELECT *
FROM Dragon
ORDER BY name, cute
""", con)

,name,year,cute
0,dragon 2,2019,0
1,drogon,2011,-100
2,hiccup,2010,10


In many instances, we are only concerned with a certain number of rows in the output table (for example, wanting to find the first two dragons in the table). The `LIMIT` keyword restricts the output to a specified number of rows. It serves a function similar to that of `.head()` in `pandas`.

In [17]:
pd.read_sql_query("""
SELECT *
FROM Dragon
LIMIT 2
""", con)

,name,year,cute
0,hiccup,2010,10
1,drogon,2011,-100


The `OFFSET` keyword indicates the index at which `LIMIT` should start. In other words, we can use `OFFSET` to shift where the `LIMIT`ing begins by a specified number of rows. For example, we might care about the dragons that are at positions #2 and #3 in the table.

In [18]:
pd.read_sql_query("""
SELECT *
FROM Dragon
LIMIT 2
OFFSET 1
""", con)

,name,year,cute
0,drogon,2011,-100
1,dragon 2,2019,0


Let's summarize what we've learned so far. We know that `SELECT` and `FROM` are the fundamental building blocks of any SQL query. We can augment these two keywords with additional clauses to refine the data in our output table. 

Any clauses that we include must follow a strict ordering within the query:

    SELECT <column list>
    FROM <table>
    [WHERE <predicate>]
    [ORDER BY <column list>]
    [LIMIT <number of rows>]
    [OFFSET <number of rows>]
    
Here, any clause contained in square brackets `[ ]` is optional – we only need to use the keyword if it is relevant to the table operation we want to perform. Also note that by convention, we use all caps for keywords in SQL statements and use newlines to make code more readable.

## Aggregating with `GROUP BY`

At this point, we've seen that SQL offers much of the same functionality that was given to us by `pandas`. We can extract data from a table, filter it, and reorder it to suit our needs.

In `pandas`, much of our analysis work relied heavily on being able to use `.groupby()` to aggregate across the rows of our dataset. SQL's answer to this task is the (very conveniently named) `GROUP BY` clause. While the outputs of `GROUP BY` are similar to those of `.groupby()` – in both cases, we obtain an output table where some column has been used for grouping – the syntax and logic used to group data in SQL are fairly different to the `pandas` implementation.

To illustrate `GROUP BY`, we will consider the `Dish` table from the `basic_examples.db` database.

In [19]:
pd.read_sql_query("""
SELECT * 
FROM Dish
""", con)

,name,type,cost
0,ravioli,entree,10
1,ramen,entree,13
2,taco,entree,7
3,edamame,appetizer,4
4,fries,appetizer,4
5,potsticker,appetizer,4
6,ice cream,dessert,5


Say we wanted to find the total costs of dishes of a certain `type`. To accomplish this, we would write the following code.

In [20]:
pd.read_sql_query("""
SELECT type, SUM(cost)
FROM Dish
GROUP BY type
""", con)

,type,SUM(cost)
0,appetizer,12
1,dessert,5
2,entree,30


What is going on here? The statement `GROUP BY type` tells SQL to group the data based on the value contained in the `type` column (whether a record is an appetizer, entree, or dessert). `SUM(cost)` sums up the costs of dishes in each `type` and displays the result in the output table.

You may be wondering: why does `SUM(cost)` come before the command to `GROUP BY type`? Don't we need to form groups before we can count the number of entries in each?

Remember that SQL is a *declarative* programming language – a SQL programmer simply states what end result they would like to see, and leaves the task of figuring out *how* to obtain this result to SQL itself. This means that SQL queries sometimes don't follow what a reader sees as a "logical" sequence of thought. Instead, SQL requires that we follow its set order of operations when constructing queries. So long as we follow this ordering, SQL will handle the underlying logic.

In practical terms: our goal with this query was to output the total `cost`s of each `type`. To communicate this to SQL, we say that we want to `SELECT` the `SUM`med `cost` values for each `type` group. 

There are many aggregation functions that can be used to aggregate the data contained in each group. Some common examples are:

* `COUNT`: count the number of rows associated with each group
* `MIN`: find the minimum value of each group
* `MAX`: find the maximum value of each group
* `SUM`: sum across all records in each group
* `AVG`: find the average value of each group

We can easily compute multiple aggregations, all at once (a task that was very tricky in `pandas`).

In [21]:
pd.read_sql_query("""
SELECT type, SUM(cost), MIN(cost), MAX(name)
FROM Dish
GROUP BY type
""", con)

,type,SUM(cost),MIN(cost),MAX(name)
0,appetizer,12,4,potsticker
1,dessert,5,5,ice cream
2,entree,30,7,taco


To count the number of rows associated with each group, we use the `COUNT` keyword. Calling `COUNT(*)` will compute the total number of rows in each group, including rows with null values. Its `pandas` equivalent is `.groupby().size()`.

In [22]:
pd.read_sql_query("""
SELECT type, COUNT(*)
FROM Dish
GROUP BY type
""", con)

,type,COUNT(*)
0,appetizer,3
1,dessert,1
2,entree,3


To exclude `NULL` values when counting the rows in each group, we explicitly call `COUNT` on a column in the table. This is similar to calling `.groupby().count()` in `pandas`.

In [23]:
pd.read_sql_query("""
SELECT year, COUNT(cute)
FROM Dragon
GROUP BY year
""", con)

,year,COUNT(cute)
0,2010,1
1,2011,1
2,2019,1


With this definition of `GROUP BY` in hand, let's update our SQL order of operations. Remember: *every* SQL query must list clauses in this order. 

    SELECT <column expression list>
    FROM <table>
    [WHERE <predicate>]
    [GROUP BY <column list>]
    [ORDER BY <column list>]
    [LIMIT <number of rows>]
    [OFFSET <number of rows>];

Note that we can use the `AS` keyword to rename columns during the selection process and that column expressions may include aggregation functions (`MAX`, `MIN`, etc.).